# 🌿 Plant Disease Detection — GPU Training on Google Colab

**Model:** MobileNetV2 (ImageNet pretrained)  
**Dataset:** PlantVillage (19 classes, ~22,810 images)  
**Strategy:** Feature extraction (frozen base) → Fine-tuning  
**Output:** `plant_model.keras` downloaded to your local machine

---
### ✅ Before running:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Upload your dataset ZIP to Google Drive (or upload directly in Cell 2)
3. Run all cells in order


## Step 1 — Verify GPU

In [2]:
import tensorflow as tf

print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU detected: {gpus[0].name}')
else:
    print('⚠️  No GPU found — go to Runtime → Change runtime type → T4 GPU')

ModuleNotFoundError: No module named 'tensorflow'

## Step 2 — Mount Google Drive & Set Dataset Path

> **Option A (recommended):** Upload your `plantvillage dataset` folder to Google Drive first,  
> then update `DRIVE_DATASET_PATH` below to match its location.
>
> **Option B:** Upload the ZIP directly in the next cell (slower, resets on session restart).

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ─── UPDATE THIS PATH to where your dataset lives in Drive ───────────────────
# Example: if you put it at MyDrive/plantvillage dataset/color/
DRIVE_DATASET_PATH = '/content/drive/MyDrive/plantvillage dataset/color'
# ─────────────────────────────────────────────────────────────────────────────

if os.path.isdir(DRIVE_DATASET_PATH):
    classes = [d for d in os.listdir(DRIVE_DATASET_PATH) if os.path.isdir(os.path.join(DRIVE_DATASET_PATH, d))]
    print(f'✅ Dataset found: {len(classes)} classes detected')
    print('Classes:', sorted(classes))
else:
    print(f'❌ Dataset NOT found at: {DRIVE_DATASET_PATH}')
    print('Please update DRIVE_DATASET_PATH above.')

ModuleNotFoundError: No module named 'google.colab'

## Step 2B — (Alternative) Upload ZIP directly to Colab
Skip this cell if you used Google Drive above.

In [ ]:
# ── SKIP THIS CELL IF YOU USED GOOGLE DRIVE ABOVE ──

# from google.colab import files
# import zipfile
#
# uploaded = files.upload()  # upload your dataset ZIP
# zip_name = list(uploaded.keys())[0]
# with zipfile.ZipFile(zip_name, 'r') as z:
#     z.extractall('/content/plantvillage')
# DRIVE_DATASET_PATH = '/content/plantvillage/color'  # adjust if needed
# print('Extracted to:', DRIVE_DATASET_PATH)

## Step 3 — Configuration

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────
DATASET_DIR      = DRIVE_DATASET_PATH
MODEL_SAVE_PATH  = '/content/plant_model.keras'    # saved here after training

TARGET_SIZE      = (224, 224)   # full resolution for best accuracy
BATCH_SIZE       = 32           # GPU can handle larger batches
EPOCHS_PHASE1    = 15           # feature extraction (frozen base)
EPOCHS_PHASE2    = 10           # fine-tuning (unfreeze top layers)
VALIDATION_SPLIT = 0.2

print('Config set ✅')
print(f'  Image size  : {TARGET_SIZE}')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Phase 1 epochs (frozen) : {EPOCHS_PHASE1}')
print(f'  Phase 2 epochs (fine-tune): {EPOCHS_PHASE2}')

## Step 4 — Data Generators

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Training generator — with augmentation for better generalisation
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=VALIDATION_SPLIT,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
)

# Validation generator — only rescale, no augmentation
val_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=VALIDATION_SPLIT,
)

train_gen = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42,
)

val_gen = val_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42,
)

class_names = list(train_gen.class_indices.keys())
num_classes  = len(class_names)

print(f'✅ Classes detected   : {num_classes}')
print(f'   Training images   : {train_gen.samples}')
print(f'   Validation images : {val_gen.samples}')
print(f'   Classes           : {class_names}')

## Step 5 — Save Class Names (for inference later)

In [ ]:
import json

class_indices = train_gen.class_indices
# Invert: index → class name
index_to_class = {v: k for k, v in class_indices.items()}

with open('/content/class_names.json', 'w') as f:
    json.dump(index_to_class, f, indent=2)

print('✅ class_names.json saved to /content/class_names.json')
print(json.dumps(index_to_class, indent=2))

## Step 6 — Build Model (MobileNetV2 + Head)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, Input

# 1. Load base model (frozen)
base_model = MobileNetV2(
    input_shape=(*TARGET_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

# 2. Build classification head
inputs  = Input(shape=(*TARGET_SIZE, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
x       = layers.Dense(128, activation='relu')(x)
x       = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs, name='plant_disease_mobilenetv2')

# 3. Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## Step 7 — Phase 1: Feature Extraction (Frozen Base, 15 Epochs)

In [ ]:
callbacks_p1 = [
    tf.keras.callbacks.ModelCheckpoint(
        '/content/best_phase1.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-7, verbose=1
    ),
]

print('=== PHASE 1: Feature Extraction (frozen base) ===')
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks_p1,
    verbose=1,
)

p1_train_acc = history1.history['accuracy'][-1]
p1_val_acc   = history1.history['val_accuracy'][-1]
print(f'\nPhase 1 complete ✅')
print(f'  Training accuracy   : {p1_train_acc:.4f} ({p1_train_acc*100:.2f}%)')
print(f'  Validation accuracy : {p1_val_acc:.4f} ({p1_val_acc*100:.2f}%)')

## Step 8 — Phase 2: Fine-Tuning (Unfreeze Top 30 Layers, 10 Epochs)

In [ ]:
# Unfreeze top 30 layers of the base model for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with a much lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_p2 = [
    tf.keras.callbacks.ModelCheckpoint(
        '/content/best_phase2.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=2,
        min_lr=1e-8, verbose=1
    ),
]

print('=== PHASE 2: Fine-Tuning (top 30 layers unfrozen) ===')
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks_p2,
    verbose=1,
)

p2_train_acc = history2.history['accuracy'][-1]
p2_val_acc   = history2.history['val_accuracy'][-1]
print(f'\nPhase 2 complete ✅')
print(f'  Training accuracy   : {p2_train_acc:.4f} ({p2_train_acc*100:.2f}%)')
print(f'  Validation accuracy : {p2_val_acc:.4f} ({p2_val_acc*100:.2f}%)')

## Step 9 — Save Final Model

In [ ]:
model.save(MODEL_SAVE_PATH)
print(f'✅ Final model saved to: {MODEL_SAVE_PATH}')

# Also save to Drive for persistence across sessions
import shutil
drive_model_path = '/content/drive/MyDrive/plant_model.keras'
shutil.copy(MODEL_SAVE_PATH, drive_model_path)
print(f'✅ Backup saved to Google Drive: {drive_model_path}')

drive_classes_path = '/content/drive/MyDrive/class_names.json'
shutil.copy('/content/class_names.json', drive_classes_path)
print(f'✅ Class names saved to Drive: {drive_classes_path}')

## Step 10 — Plot Training History

In [ ]:
import matplotlib.pyplot as plt

# Combine both phases
acc     = history1.history['accuracy']     + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss    = history1.history['loss']         + history2.history['loss']
val_loss= history1.history['val_loss']     + history2.history['val_loss']

phase1_end = len(history1.history['accuracy'])
epochs_range = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(epochs_range, acc,     label='Train Accuracy',      color='royalblue')
ax1.plot(epochs_range, val_acc, label='Validation Accuracy', color='darkorange')
ax1.axvline(phase1_end, color='gray', linestyle='--', alpha=0.7, label='Fine-tune start')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss plot
ax2.plot(epochs_range, loss,     label='Train Loss',      color='royalblue')
ax2.plot(epochs_range, val_loss, label='Validation Loss', color='darkorange')
ax2.axvline(phase1_end, color='gray', linestyle='--', alpha=0.7, label='Fine-tune start')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Plant Disease Detection — Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to /content/training_history.png')

## Step 11 — Final Summary

In [ ]:
print('=' * 60)
print('  TRAINING COMPLETE — FINAL SUMMARY')
print('=' * 60)
print(f'  Number of classes        : {num_classes}')
print(f'  Training images          : {train_gen.samples}')
print(f'  Validation images        : {val_gen.samples}')
print(f'  Phase 1 val accuracy     : {p1_val_acc*100:.2f}%')
print(f'  Phase 2 val accuracy     : {p2_val_acc*100:.2f}%  ← final')
print(f'  Model saved              : {MODEL_SAVE_PATH}')
print(f'  Drive backup             : /content/drive/MyDrive/plant_model.keras')
print('=' * 60)

## Step 12 — Download Model to Local Machine

In [ ]:
from google.colab import files

# Download the trained model
print('Downloading plant_model.keras ...')
files.download('/content/plant_model.keras')

# Download class names JSON
print('Downloading class_names.json ...')
files.download('/content/class_names.json')

# Download training plot
print('Downloading training_history.png ...')
files.download('/content/training_history.png')

print('\n✅ All files downloaded!')
print('Place plant_model.keras → plant_disease_prediction/models/')
print('Place class_names.json  → plant_disease_prediction/models/')